# 02 — Data Download Verification
**Project:** Real-Time Audio Fall Detection (TinyML)

This notebook systematically verifies that all datasets downloaded by
`01_data_download.ipynb` are present and intact on Google Drive.

All expected values are **source-verified** — gathered from the original dataset
pages (Zenodo API, Kaggle, Mendeley, GitHub). See `data/DATASET_MANIFEST.md` for
full documentation.

### Checks performed
| Check | Description |
|-------|-------------|
| Exists | Folder exists and is non-empty |
| File count | Number of files matches source-verified expected count |
| Size | Total size is within source-verified expected range |
| Media integrity | A sample of audio files can be opened and decoded |
| Empty / truncated files | Detect 0-byte or suspiciously small files |
| Expected sub-structure | Key sub-folders or files are present |
| FSD50K split-zip | Detects unextracted split-zip archives with fix instructions |
| MD5 checksums | Verifies Zenodo archive integrity against published hashes |

## 1 · Setup

In [ ]:
from google.colab import drive
import os, random, subprocess
from pathlib import Path
from collections import Counter

drive.mount("/content/drive")

# Install soundfile for audio validation
subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
import soundfile as sf

# ── Project paths (must match 01_data_download) ──
BASE_DIR = "/content/drive/MyDrive/DL-Project"
RAW_DIR  = os.path.join(BASE_DIR, "raw")

DATASETS = ["DESED", "CAUCAFall", "WaterLeakage", "HumanScreaming", "FSD50K", "SAFE", "AudioSet"]

print(f"Raw data directory: {RAW_DIR}")
print(f"Exists: {os.path.isdir(RAW_DIR)}")

In [ ]:
# ── Helper functions ───────────────────────────────────────────

def count_files(directory):
    """Recursively count files."""
    return sum(1 for _ in Path(directory).rglob("*") if _.is_file())


def dir_size_mb(directory):
    """Total size in MB."""
    return sum(f.stat().st_size for f in Path(directory).rglob("*") if f.is_file()) / (1024 * 1024)


def get_extension_counts(directory):
    """Count files by extension."""
    exts = Counter()
    for f in Path(directory).rglob("*"):
        if f.is_file():
            exts[f.suffix.lower()] += 1
    return exts


def find_empty_or_tiny_files(directory, min_bytes=100):
    """Find files that are 0-byte or smaller than min_bytes."""
    bad = []
    for f in Path(directory).rglob("*"):
        if f.is_file() and f.stat().st_size < min_bytes:
            bad.append((str(f), f.stat().st_size))
    return bad


def find_leftover_archives(directory):
    """Find .zip / .tar.gz files that may not have been extracted."""
    archives = []
    for ext in ("*.zip", "*.tar.gz", "*.tgz", "*.tar"):
        archives.extend(Path(directory).rglob(ext))
    return [str(a) for a in archives]


AUDIO_EXTS = {".wav", ".flac", ".ogg", ".mp3", ".aac", ".m4a", ".wma", ".aiff"}

def collect_audio_files(directory):
    """Collect all audio file paths."""
    return [f for f in Path(directory).rglob("*")
            if f.is_file() and f.suffix.lower() in AUDIO_EXTS]


def sample_audio_check(directory, n_samples=20):
    """Try to open a random sample of audio files with soundfile.
    Returns (n_ok, n_fail, failures_list).
    """
    audio_files = collect_audio_files(directory)
    if not audio_files:
        return 0, 0, []

    sample = random.sample(audio_files, min(n_samples, len(audio_files)))
    ok, fail, failures = 0, 0, []

    for fp in sample:
        try:
            info = sf.info(str(fp))
            if info.frames == 0:
                fail += 1
                failures.append((str(fp), "0 frames"))
            else:
                ok += 1
        except Exception as e:
            fail += 1
            failures.append((str(fp), str(e)[:80]))

    return ok, fail, failures


print("Verification helpers loaded.")

## 2 · Expected dataset profiles

All values below are **source-verified** — gathered from the original dataset
pages (Zenodo API, Kaggle, Mendeley, GitHub) on 2025-03-06.
See `data/DATASET_MANIFEST.md` for full documentation including MD5 checksums.

In [ ]:
# ── Source-verified dataset profiles ────────────────────────────
# All values cross-referenced with original dataset pages.
# See data/DATASET_MANIFEST.md for full details and MD5 checksums.
#
# min_files  : hard lower-bound from source (below → something is wrong)
# min_mb     : minimum total size in MB from source
# media_exts : expected dominant file extensions
# key_items  : sub-folders or files that MUST be present
# exact_files: exact expected file count (None if variable)
# notes      : special considerations for this dataset

EXPECTED = {
    "DESED": {
        "min_files": 50,
        "min_mb": 5_000,
        "media_exts": {".wav"},
        "key_items": ["soundbank_validation.tsv"],
        "exact_files": None,  # Varies depending on which archives were extracted
        "source": "Zenodo — https://zenodo.org/records/4560938",
        "source_size": "10.1 GB (6 archive files)",
        "notes": "Soundbank + eval archives extract to thousands of WAV files.",
    },
    "CAUCAFall": {
        "min_files": 100,
        "min_mb": 6_000,
        "media_exts": {".avi", ".png", ".txt"},
        "key_items": [],
        "exact_files": None,
        "source": "Mendeley — https://data.mendeley.com/datasets/7w7fccy7ky/4",
        "source_size": "7.75 GB (single ZIP)",
        "notes": (
            "VIDEO dataset (not audio). Contains AVI videos, PNG frames, "
            "TXT segmentation labels. 10 subjects × 10 activities."
        ),
    },
    "WaterLeakage": {
        "min_files": 30,
        "min_mb": 5,
        "media_exts": {".wav"},
        "key_items": [
            "dataset",
            "dataset/Labelled",
            "dataset/Labelled/Train",
            "dataset/Labelled/Test",
            "dataset/Unlabelled",
            "README.md",
        ],
        "exact_files": None,
        "source": "GitHub — aliEmreOzturk/water_leakage_voice_data",
        "source_size": "Small (git clone)",
        "notes": (
            "11 class directories in Train/ and Test/ "
            "(11,12,13,21,22,23,31,32,33,fs,hw). ~30 unlabelled WAVs."
        ),
    },
    "HumanScreaming": {
        "min_files": 3_493,
        "min_mb": 5_400,
        "media_exts": {".wav"},
        "key_items": ["Screaming", "NotScreaming"],
        "exact_files": 3_493,
        "source": "Kaggle — whats2000/human-screaming-detection-dataset",
        "source_size": "5.68 GB (Version 2)",
        "notes": "2,631 NotScreaming + 862 Screaming WAVs. 44.1 kHz, 1-10s each.",
    },
    "FSD50K": {
        "min_files": 51_197,
        "min_mb": 20_000,
        "media_exts": {".wav"},
        "key_items": [
            "FSD50K.dev_audio",
            "FSD50K.eval_audio",
            "FSD50K.ground_truth",
            "FSD50K.metadata",
            "FSD50K.doc",
        ],
        "exact_files": None,  # 51,197 audio + metadata/doc files
        "source": "Zenodo — https://zenodo.org/records/4060432",
        "source_size": "~25.7 GB (11 archive files, split-zip)",
        "notes": (
            "51,197 clips: 40,966 dev + 10,231 eval. PCM 16-bit 44.1 kHz mono. "
            "Archives use SPLIT-ZIP — must merge with `zip -s 0` before unzip."
        ),
    },
    "SAFE": {
        "min_files": 950,
        "min_mb": 260,
        "media_exts": {".wav"},
        "key_items": [],
        "exact_files": 950,
        "source": "Kaggle — antonygarciag/fall-audio-detection-dataset",
        "source_size": "273.63 MB (Version 5)",
        "notes": "475 fall + 475 non-fall WAVs. 48 kHz, 3s clips. Flat directory.",
    },
    "AudioSet": {
        "min_files": 10,
        "min_mb": 1,
        "media_exts": {".wav"},
        "key_items": [],
        "exact_files": None,  # Depends on YouTube availability
        "source": "YouTube via yt-dlp (filtered AudioSet segments)",
        "source_size": "Variable (up to 500 segments)",
        "notes": (
            "Yield depends on YouTube video availability. "
            "Expect 10-50% failure rate. Realistic: 250-450 clips."
        ),
    },
}

print(f"Source-verified profiles defined for {len(EXPECTED)} datasets.\n")
for ds, exp in EXPECTED.items():
    print(f"  {ds:20s}  files >= {exp['min_files']:>6,}   size >= {exp['min_mb']:>6,} MB   ({exp['source_size']})")

## 3 · Run full verification

In [ ]:
# ── Full verification pass ─────────────────────────────────────

PASS = "\033[92mPASS\033[0m"
WARN = "\033[93mWARN\033[0m"
FAIL = "\033[91mFAIL\033[0m"

# Media extensions that are NOT audio (skip audio-integrity check for these)
VIDEO_EXTS = {".avi", ".mp4", ".mkv", ".mov"}
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}

results = {}  # dataset -> {status, issues}

for ds in DATASETS:
    ds_path = os.path.join(RAW_DIR, ds)
    exp = EXPECTED[ds]
    issues = []

    print("=" * 65)
    print(f"  {ds}")
    print(f"  Source: {exp['source']}")
    print(f"  Expected: {exp['source_size']}")
    if exp.get("notes"):
        print(f"  Note: {exp['notes']}")
    print("-" * 65)

    # ── Check 1: Folder exists ──
    if not os.path.isdir(ds_path):
        print(f"  {FAIL}  Folder does not exist: {ds_path}")
        issues.append("MISSING folder")
        results[ds] = {"status": "FAIL", "issues": issues}
        print()
        continue

    # ── Check 2: File count ──
    n_files = count_files(ds_path)
    if n_files == 0:
        print(f"  {FAIL}  Folder is EMPTY")
        issues.append("EMPTY folder")
        results[ds] = {"status": "FAIL", "issues": issues}
        print()
        continue

    exact = exp.get("exact_files")
    if exact and n_files != exact:
        print(f"  {WARN}  File count: {n_files:,}  (expected exactly {exact:,})")
        issues.append(f"File count mismatch: {n_files:,} != {exact:,}")
    elif n_files < exp["min_files"]:
        print(f"  {WARN}  File count: {n_files:,}  (expected >= {exp['min_files']:,})")
        issues.append(f"Low file count: {n_files:,} < {exp['min_files']:,}")
    else:
        print(f"  {PASS}  File count: {n_files:,}  (expected >= {exp['min_files']:,})")

    # ── Check 3: Total size ──
    size = dir_size_mb(ds_path)
    if size < exp["min_mb"]:
        print(f"  {WARN}  Total size: {size:,.1f} MB  (expected >= {exp['min_mb']:,} MB)")
        issues.append(f"Small size: {size:,.1f} MB < {exp['min_mb']:,} MB")
    else:
        print(f"  {PASS}  Total size: {size:,.1f} MB  (expected >= {exp['min_mb']:,} MB)")

    # ── Check 4: Extension distribution ──
    ext_counts = get_extension_counts(ds_path)
    print(f"  INFO  Extensions: ", end="")
    for ext, cnt in ext_counts.most_common(8):
        print(f"{ext}={cnt}  ", end="")
    print()

    # Determine if this is an audio or video/image dataset
    is_audio_dataset = not (exp["media_exts"] & (VIDEO_EXTS | IMAGE_EXTS))

    if is_audio_dataset:
        audio_files = collect_audio_files(ds_path)
        if len(audio_files) == 0:
            print(f"  {WARN}  No audio files found (expected {exp['media_exts']})")
            issues.append("No audio files found")
        else:
            print(f"  {PASS}  Audio files: {len(audio_files):,}")
    else:
        # For video/image datasets, just check media_exts are present
        found_exts = set(ext_counts.keys()) & exp["media_exts"]
        if found_exts:
            print(f"  {PASS}  Expected media types present: {found_exts}")
        else:
            print(f"  {WARN}  Expected media types {exp['media_exts']} not found")
            issues.append(f"Missing expected media types: {exp['media_exts']}")

    # ── Check 5: Empty / tiny files ──
    tiny = find_empty_or_tiny_files(ds_path)
    if tiny:
        print(f"  {WARN}  Found {len(tiny)} empty / tiny file(s) (<100 bytes):")
        for fpath, sz in tiny[:5]:
            print(f"         {os.path.basename(fpath)}  ({sz} bytes)")
        if len(tiny) > 5:
            print(f"         ... and {len(tiny) - 5} more")
        issues.append(f"{len(tiny)} empty/tiny files")
    else:
        print(f"  {PASS}  No empty/tiny files")

    # ── Check 6: Leftover archives ──
    archives = find_leftover_archives(ds_path)
    if archives:
        print(f"  {WARN}  Leftover archives (may not have been extracted):")
        for a in archives[:5]:
            print(f"         {os.path.basename(a)}")
        if len(archives) > 5:
            print(f"         ... and {len(archives) - 5} more")
        issues.append(f"{len(archives)} leftover archive(s)")
    else:
        print(f"  {PASS}  No leftover archives")

    # ── Check 7: Audio integrity (sample) — only for audio datasets ──
    if is_audio_dataset:
        audio_files_for_check = collect_audio_files(ds_path)
        if audio_files_for_check:
            n_ok, n_fail, failures = sample_audio_check(ds_path, n_samples=20)
            total_checked = n_ok + n_fail
            if n_fail > 0:
                print(f"  {WARN}  Audio sample check: {n_ok}/{total_checked} OK, {n_fail} FAILED")
                for fpath, err in failures[:3]:
                    print(f"         {os.path.basename(fpath)}: {err}")
                issues.append(f"{n_fail}/{total_checked} audio samples unreadable")
            else:
                print(f"  {PASS}  Audio sample check: {n_ok}/{total_checked} OK")

    # ── Check 8: Key items ──
    for item in exp.get("key_items", []):
        item_path = os.path.join(ds_path, item)
        if os.path.exists(item_path):
            print(f"  {PASS}  Key item present: {item}")
        else:
            print(f"  {WARN}  Key item MISSING: {item}")
            issues.append(f"Missing key item: {item}")

    # ── Verdict ──
    if not issues:
        results[ds] = {"status": "PASS", "issues": []}
    elif any("EMPTY" in i or "MISSING" in i for i in issues):
        results[ds] = {"status": "FAIL", "issues": issues}
    else:
        results[ds] = {"status": "WARN", "issues": issues}

    print()

## 3b · FSD50K split-zip check

FSD50K uses **split-zip** archives (`.z01`–`.z05` + `.zip`). If only the archive
parts exist without extracted directories, the download succeeded but extraction
was incomplete.

In [ ]:
# ── FSD50K split-zip detection ──────────────────────────────────
# If archive parts (.z01-.z05) exist but extracted dirs don't,
# the user needs to merge and extract manually.

fsd_path = os.path.join(RAW_DIR, "FSD50K")

if os.path.isdir(fsd_path):
    # Check for un-extracted split-zip parts
    split_parts = list(Path(fsd_path).glob("*.z0*"))
    zip_files   = list(Path(fsd_path).glob("*.zip"))

    has_dev_audio  = os.path.isdir(os.path.join(fsd_path, "FSD50K.dev_audio"))
    has_eval_audio = os.path.isdir(os.path.join(fsd_path, "FSD50K.eval_audio"))

    if split_parts and not (has_dev_audio and has_eval_audio):
        print(f"\033[91m{'='*65}\033[0m")
        print(f"\033[91m  FSD50K: SPLIT-ZIP ARCHIVES DETECTED BUT NOT EXTRACTED\033[0m")
        print(f"\033[91m{'='*65}\033[0m")
        print(f"  Found {len(split_parts)} split-zip parts + {len(zip_files)} zip files")
        print(f"  But extracted directories are missing:")
        print(f"    FSD50K.dev_audio/  : {'EXISTS' if has_dev_audio else 'MISSING'}")
        print(f"    FSD50K.eval_audio/ : {'EXISTS' if has_eval_audio else 'MISSING'}")
        print()
        print("  To fix, run these commands in a Colab cell:")
        print("  ─────────────────────────────────────────────")
        print(f"  %cd {fsd_path}")
        print()
        if not has_dev_audio:
            print("  # Merge and extract dev audio (6 parts)")
            print("  !zip -s 0 FSD50K.dev_audio.zip --out unsplit_dev.zip")
            print("  !unzip unsplit_dev.zip")
            print("  !rm unsplit_dev.zip")
            print()
        if not has_eval_audio:
            print("  # Merge and extract eval audio (2 parts)")
            print("  !zip -s 0 FSD50K.eval_audio.zip --out unsplit_eval.zip")
            print("  !unzip unsplit_eval.zip")
            print("  !rm unsplit_eval.zip")
        print()
    elif has_dev_audio and has_eval_audio:
        dev_count  = count_files(os.path.join(fsd_path, "FSD50K.dev_audio"))
        eval_count = count_files(os.path.join(fsd_path, "FSD50K.eval_audio"))
        print(f"  FSD50K extraction looks good:")
        print(f"    FSD50K.dev_audio/  : {dev_count:,} files (expected 40,966)")
        print(f"    FSD50K.eval_audio/ : {eval_count:,} files (expected 10,231)")
        if dev_count < 40_966:
            print(f"    \033[93mWARN: dev_audio has fewer files than expected\033[0m")
        if eval_count < 10_231:
            print(f"    \033[93mWARN: eval_audio has fewer files than expected\033[0m")
    else:
        print("  FSD50K folder exists but no archives or extracted dirs found.")
        print("  Re-run the FSD50K download cell in 01_data_download.ipynb.")
else:
    print("  FSD50K folder does not exist — skipping split-zip check.")

## 3c · MD5 checksum verification (Zenodo datasets)

DESED and FSD50K archives have **MD5 checksums published by Zenodo**. If any
archive files still exist on Drive (before extraction deletes them), this cell
verifies them against the official values.

> **Note:** This only applies if archive files are still present. If they were
> already extracted and deleted, this cell will simply skip them.

In [ ]:
# ── MD5 checksum verification for Zenodo archives ──────────────
import hashlib

# Official MD5 checksums from Zenodo API responses.
ZENODO_CHECKSUMS = {
    "DESED": {
        "DESED_synth_soundbank.tar.gz":              "03b51e3506ae28157a26101748045e90",
        "DESED_synth_eval_dcase2019.tar.gz":         "e1aad0a714bb98d2b58f3d62122077b8",
        "DESED_synth_dcase2019jams.tar.gz":          "e5d6348d9b9ca19d08b7afba0e987de3",
        "DESED_synth_dcase20_train_val_jams.tar.gz": "01f2ba4e33c82006d8e407b75f103fe7",
        "DESED_synth_dcase20_eval_jams.tar.gz":      "105774e4528b266c829f3a6fdad4397d",
        "soundbank_validation.tsv":                  "2eba5a6fe230baecc1803dab526a77a5",
    },
    "FSD50K": {
        "FSD50K.dev_audio.z01":      "faa7cf4cc076fc34a44a479a5ed862a3",
        "FSD50K.dev_audio.z02":      "8f9b66153e68571164fb1315d00bc7bc",
        "FSD50K.dev_audio.z03":      "1196ef47d267a993d30fa98af54b7159",
        "FSD50K.dev_audio.z04":      "d088ac4e11ba53daf9f7574c11cccac9",
        "FSD50K.dev_audio.z05":      "81356521aa159accd3c35de22da28c7f",
        "FSD50K.dev_audio.zip":      "c480d119b8f7a7e32fdb58f3ea4d6c5a",
        "FSD50K.eval_audio.z01":     "3090670eaeecc013ca1ff84fe4442aeb",
        "FSD50K.eval_audio.zip":     "6fa47636c3a3ad5c7dfeba99f2637982",
        "FSD50K.ground_truth.zip":   "ca27382c195e37d2269c4c866dd73485",
        "FSD50K.metadata.zip":       "b9ea0c829a411c1d42adb9da539ed237",
        "FSD50K.doc.zip":            "3516162b82dc2945d3e7feba0904e800",
    },
}


def md5_file(filepath, chunk_size=8192 * 1024):
    """Compute MD5 hash of a file, reading in chunks to conserve memory."""
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


print("=" * 65)
print("  MD5 CHECKSUM VERIFICATION (Zenodo archives)")
print("=" * 65)

checked = 0
passed  = 0
failed  = 0

for ds, checksums in ZENODO_CHECKSUMS.items():
    ds_path = os.path.join(RAW_DIR, ds)
    if not os.path.isdir(ds_path):
        continue

    print(f"\n  {ds}/")
    found_any = False

    for filename, expected_md5 in checksums.items():
        filepath = os.path.join(ds_path, filename)
        if not os.path.isfile(filepath):
            continue

        found_any = True
        checked += 1
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"    Computing MD5 for {filename} ({size_mb:,.1f} MB) ...", end=" ", flush=True)

        actual_md5 = md5_file(filepath)
        if actual_md5 == expected_md5:
            print(f"\033[92mOK\033[0m")
            passed += 1
        else:
            print(f"\033[91mMISMATCH\033[0m")
            print(f"      Expected: {expected_md5}")
            print(f"      Got:      {actual_md5}")
            failed += 1

    if not found_any:
        print(f"    (no archive files found — likely already extracted)")

print(f"\n  Summary: {checked} files checked, {passed} passed, {failed} failed")
if checked == 0:
    print("  (All archives were already extracted — this is normal.)")
elif failed > 0:
    print("  \033[91mSome archives are corrupted! Re-download those files.\033[0m")
else:
    print("  \033[92mAll archive checksums match.\033[0m")

## 4 · Summary report

In [ ]:
# ── Final summary ──────────────────────────────────────────────

STATUS_ICON = {"PASS": "\u2705", "WARN": "\u26A0\uFE0F", "FAIL": "\u274C"}

print("=" * 65)
print("  VERIFICATION SUMMARY")
print("=" * 65)

all_ok = True
for ds in DATASETS:
    r = results.get(ds, {"status": "FAIL", "issues": ["Not checked"]})
    icon = STATUS_ICON.get(r["status"], "?")
    line = f"  {icon}  {ds:20s}  {r['status']}"
    if r["issues"]:
        line += f"  — {'; '.join(r['issues'])}"
        all_ok = False
    print(line)

print("=" * 65)

if all_ok:
    print("\n  All datasets verified successfully!")
    print("  You can proceed to the preprocessing step.")
else:
    print("\n  Some datasets have issues. Review the warnings/failures above.")
    print("  Re-run the corresponding cell in 01_data_download.ipynb to fix.")

## 5 · Detailed directory tree (optional)

Run the cell below to print the top-level structure of each dataset folder.

In [ ]:
# ── Directory tree (depth=2) ────────────────────────────────────

for ds in DATASETS:
    ds_path = os.path.join(RAW_DIR, ds)
    print(f"\n{'=' * 50}")
    print(f"  {ds}/")
    print(f"{'=' * 50}")

    if not os.path.isdir(ds_path):
        print("  (does not exist)")
        continue

    ds_root = Path(ds_path)
    # Show top-level items
    items = sorted(ds_root.iterdir())
    for item in items:
        if item.is_dir():
            sub_count = count_files(item)
            print(f"  {item.name}/  ({sub_count} files)")
            # Show second level
            for sub in sorted(item.iterdir())[:10]:
                if sub.is_dir():
                    print(f"    {sub.name}/  ({count_files(sub)} files)")
                else:
                    print(f"    {sub.name}  ({sub.stat().st_size / 1024:.0f} KB)")
            sub_items = list(item.iterdir())
            if len(sub_items) > 10:
                print(f"    ... and {len(sub_items) - 10} more items")
        else:
            print(f"  {item.name}  ({item.stat().st_size / 1024:.0f} KB)")

print("\nDone.")